# TRACK-FA Merge and Quality Control

This notebook prepares TRACK-FA paired data for progression-biomarker analysis.

Pipeline:
1. Load TRACK-FA source tables and project configuration.
2. Merge clinical, demographic, and imaging data into paired visit records.
3. Validate the long-format visit table and paired-delta table.
4. Check imaging coverage, subject identifier integrity, and leave-one-group-out grouping.
5. Summarize clinical change and list final feature names by category.


In [1]:
# Setup: load dependencies, configuration, and data paths.

import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not find the project root containing src/ and data/.")


_REPO_ROOT = find_project_root(Path.cwd())
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

import pandas as pd

from src.data.trackfa import (
    run_merge,
    qc_long,
    qc_pairs,
    imaging_coverage,
    dry_run_logo_split,
    delta_summary_table,
)


In [2]:
# Run the TRACK-FA merge and display paired analysis tables.

long_df, pairs_df = run_merge(save=False, verbose=True)
print('trackfa_long:', long_df.shape)
print('trackfa_pairs:', pairs_df.shape)

display(long_df.head())
display(pairs_df.head())


[trackfa_pairs] dropped 2 rows with missing clinical values (before=298, after=296). Removed by pair: {'V1V2': 1, 'V2V3': 1}
[trackfa_pairs] dropped 222 rows failing strict sheet completeness (before=296, after=74). Removed by pair: {'V1V2': 114, 'V2V3': 108}
[trackfa_pairs] failed-sheet counts (rows where this condition fails): POMs_complete_baseline=147, POMs_complete_followup=167, BrainSpineMorph_complete_baseline=29, BrainSpineMorph_complete_followup=26, BrainDTI_complete_baseline=42, BrainDTI_complete_followup=40
[trackfa_pairs] top missing columns among removed rows:
delta_tNAA_myo_Ins       0.653153
delta_DN_suscept         0.621622
delta_DN_vol             0.621622
tNAA_myo_Ins_followup    0.527027
DN_suscept_baseline      0.436937
DN_vol_baseline          0.436937
DN_suscept_followup      0.432432
DN_vol_followup          0.432432
tNAA_myo_Ins_baseline    0.400901
delta_MD_PCR             0.216216
delta_MD_Cing_h          0.216216
delta_MD_Cing            0.216216
delta_MD_EC 

,subject_id,visit,site,age,gender,gaa_1,gaa_2,onset_age,disease_duration,mfars_total,...,AD_PTR,AD_ILF_IFOF,AD_EC,AD_Cing,AD_Cing_h,AD_Fx_ST,AD_SLF,AD_SFOF,AD_UNC,AD_Tap
0,AAN001,1,2.0,26.3,2.0,612,912,15.0,11.3,27.666667,...,0.001152,0.001158,0.001028,0.001057,0.001129,0.001305,0.000962,0.000964,0.001128,0.001614
1,AAN001,2,2.0,26.3,2.0,612,912,15.0,11.3,19.000000,...,0.001176,0.001198,0.001032,0.001095,0.001105,0.001295,0.000973,0.000997,0.001126,0.001638
2,AAN001,3,2.0,26.3,2.0,612,912,15.0,11.3,29.000000,...,0.001179,0.001186,0.001029,0.001079,0.001125,0.001270,0.000980,0.001004,0.001109,0.001651
3,AAN002,1,2.0,25.6,2.0,520,930,11.0,14.6,41.000000,...,0.001228,0.001271,0.001051,0.001091,0.001101,0.001320,0.001003,0.001001,0.001127,0.001717
4,AAN002,2,2.0,25.6,2.0,520,930,11.0,14.6,51.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,patient_id,site,age,gender,gaa_1,gaa_2,onset_age,disease_duration,mfars_total_baseline,mfars_total_followup,...,delta_AD_SLF,AD_SFOF_baseline,AD_SFOF_followup,delta_AD_SFOF,AD_UNC_baseline,AD_UNC_followup,delta_AD_UNC,AD_Tap_baseline,AD_Tap_followup,delta_AD_Tap
0,AAN005_V1V2,2.0,32.6,1.0,512,712,21.0,11.6,23.333333,39.500000,...,0.000001,0.001019,0.001042,0.000023,0.001135,0.001114,-2.122000e-05,0.001552,0.001542,-0.000010
1,AAN006_V1V2,2.0,13.8,1.0,660,830,6.0,7.8,28.000000,29.000000,...,-0.000031,0.001202,0.001174,-0.000029,0.001270,0.001232,-3.796500e-05,0.001668,0.001678,0.000010
2,AAN009_V1V2,2.0,16.7,1.0,845,960,11.0,5.7,53.666667,54.666667,...,-0.000014,0.001085,0.001074,-0.000011,0.001096,0.001112,1.629500e-05,0.001698,0.001706,0.000008
3,AAN026_V1V2,2.0,18.0,2.0,660,760,5.0,13.0,23.666667,28.000000,...,-0.000002,0.000982,0.000979,-0.000004,0.001268,0.001248,-1.984500e-05,0.001603,0.001615,0.000012
4,AAN028_V1V2,2.0,12.8,1.0,560,900,7.0,5.8,16.000000,25.333333,...,0.000002,0.001079,0.001028,-0.000051,0.001189,0.001189,6.000000e-07,0.001633,0.001540,-0.000093


In [3]:
# Check the visit-level table for expected subjects, visits, and clinical scores.

out = qc_long(long_df)
print('rows:', out['n_rows'])
print('cols:', out['n_cols'])
print('unique subjects:', out['n_subjects'])
display(out['visit_distribution'])
display(out['missingness_by_group'])

fig = out.get('missing_heatmap_fig')
if fig is not None:
    display(fig)
else:
    print('missing heatmap unavailable:', out.get('missing_heatmap_error'))


rows: 522
cols: 161
unique subjects: 174


,n
visit,
1,174
2,174
3,174


,demo,clinical,imaging
0,0.0,0.060664,0.200404


missing heatmap unavailable: No module named 'seaborn'


In [4]:
# Check paired baseline/follow-up rows and longitudinal deltas.

out = qc_pairs(pairs_df)
display(out['pair_counts'])
print('subjects with both pair types:', out['subjects_with_both_pair_types'])
print('subjects with one pair type:', out['subjects_with_one_pair_type'])

fig = out.get('delta_hist_fig')
if fig is not None:
    display(fig)
else:
    print('delta hist unavailable:', out.get('delta_hist_error'))


,n
1,
V1V2,43
V2V3,31


subjects with both pair types: 21
subjects with one pair type: 32
delta hist unavailable: No module named 'seaborn'


In [5]:
# Summarize imaging-feature availability across the merged dataset.

coverage = imaging_coverage(long_df, missing_threshold=0.20)
flagged = coverage[coverage['flag_gt20pct_missing']].copy()
print('n imaging features:', len(coverage))
print('flagged (>20% missing):', len(flagged))
display(flagged.head(30))


n imaging features: 149
flagged (>20% missing): 115


,frac_non_nan,frac_missing,flag_gt20pct_missing
sAD_c3c5,0.781609,0.218391,True
sMD_c3c5,0.781609,0.218391,True
sRD_c3c5,0.781609,0.218391,True
sFA_c3c5,0.781609,0.218391,True
RD_MCP,0.779693,0.220307,True
MD_Tap,0.779693,0.220307,True
RD_Fx_ST,0.779693,0.220307,True
RD_Cing_h,0.779693,0.220307,True
RD_Cing,0.779693,0.220307,True
RD_EC,0.779693,0.220307,True


In [6]:
# Confirm subject-wise grouping for leave-one-group-out validation.

out = dry_run_logo_split(pairs_df)
print(out)


{'ok': True, 'n_splits': 53}


In [7]:
# Summarize follow-up-minus-baseline clinical change.

display(delta_summary_table(pairs_df))


,clinical_score,delta_mean,delta_std,SRM_mean_over_std,n
0,mfars_total,2.250000,4.936526,0.455786,74
1,adl_total,0.675676,2.570156,0.262893,74
2,sara_total,0.851351,2.519417,0.337916,74


In [8]:
# List final feature names by modality category.

from src.data.trackfa import feature_catalog

cat = feature_catalog(long_df, pairs_df)
print('Demographic columns:')
print(cat['demographic_columns'])
print('\nClinical base names:')
print(cat['clinical_base_names'])
print('\nLong-format imaging columns (n=%d):' % len(cat['long_imaging_columns']))
print(cat['long_imaging_columns'])
print('\nPairs static columns:')
print(cat['pairs_static_columns'])
print('\nPairs clinical columns:')
print(cat['pairs_clinical_columns'])
print('\nPairs imaging base names (n=%d):' % len(cat['pairs_imaging_base_names']))
print(cat['pairs_imaging_base_names'])


Demographic columns:
['site', 'age', 'gender', 'gaa_1', 'gaa_2', 'onset_age', 'disease_duration']

Clinical base names:
['mfars_total', 'adl_total', 'sara_total']

Long-format imaging columns (n=149):
['AD_ACR', 'AD_ALIC', 'AD_CP', 'AD_CST', 'AD_Cing', 'AD_Cing_h', 'AD_EC', 'AD_Fx', 'AD_Fx_ST', 'AD_ICP', 'AD_ILF_IFOF', 'AD_MCP', 'AD_PCR', 'AD_PCT', 'AD_PLIC', 'AD_PTR', 'AD_RLIC', 'AD_SCP', 'AD_SCR', 'AD_SFOF', 'AD_SLF', 'AD_Tap', 'AD_UNC', 'AD_bCC', 'AD_gCC', 'AD_mLEM', 'AD_sCC', 'Accumbens_area', 'Amygdala', 'Brain_Stem_FS', 'CCtotal', 'Caudate', 'Cereb_vol', 'Cerebellum_CerebNet', 'Cerebellum_Cortex_CerebNet', 'Cerebellum_Cortex_FS', 'Cerebellum_WM_CerebNet', 'Cerebellum_WM_FS', 'DN_suscept', 'DN_vol', 'FA_ACR', 'FA_ALIC', 'FA_CP', 'FA_CST', 'FA_Cing', 'FA_Cing_h', 'FA_EC', 'FA_Fx', 'FA_Fx_ST', 'FA_ICP', 'FA_ILF_IFOF', 'FA_MCP', 'FA_PCR', 'FA_PCT', 'FA_PLIC', 'FA_PTR', 'FA_RLIC', 'FA_SCP', 'FA_SCR', 'FA_SFOF', 'FA_SLF', 'FA_Tap', 'FA_UNC', 'FA_bCC', 'FA_gCC', 'FA_mLEM', 'FA_sCC', 'Hi